# 01. EDA raw_data: данные, модальности и ограничения

**Цель.** Проверить, что задача раннего предупреждения строится из сырых логов и KPI, а не из заранее подготовленных таблиц. На этом этапе фиксируются структура данных, покрытие модальностей и ограничения, которые должен учитывать baseline и финальная модель.

**Допущение.** Единственный источник входных данных финального контура — `raw_data/`. Все рабочие артефакты создаются заново в `dashboard/artifacts/`.


In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dashboard.final_data_pipeline import PROJECT_ROOT, SEED, scan_raw_data, write_json

random.seed(SEED)
np.random.seed(SEED)

RAW_DATA = PROJECT_ROOT / "raw_data"
ARTIFACT_DIR = PROJECT_ROOT / "dashboard" / "artifacts"
REPORT_DIR = ARTIFACT_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"raw_data: {RAW_DATA}")
print(f"artifact_dir: {ARTIFACT_DIR}")


## 1. Постановка задачи

Система должна наблюдать за запуском workload во времени и поднимать предупреждение до фактического сбоя. Поэтому полезная тревога не равна простой классификации отдельной строки лога: решение должно учитывать временное окно, доступность логов и KPI, расстояние до события и риск утечки через слишком поздние признаки.


In [ ]:
inventory, issues = scan_raw_data(RAW_DATA)
summary = (
    inventory.groupby(["pair_status", "scenario_type"], dropna=False)
    .agg(runs=("run_id", "nunique"), workloads=("workload", "nunique"))
    .reset_index()
)
display(summary)


**Локальный вывод.** Основной обучающий контур должен брать только пары `STRICT`, где есть и raw log, и KPI. Неполные сценарии важны как аудит покрытия, но они не должны незаметно попадать в supervised training.


## 2. Анатомия модальностей

Проверяем распределение запусков по workload и типам сценариев. Это нужно для split: близкие сценарии одного workload нельзя случайно смешивать между train и test.


In [ ]:
strict = inventory.loc[inventory["pair_status"].eq("STRICT")].copy()
workload_summary = (
    strict.groupby(["workload", "scenario_type"])["run_id"]
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
)
workload_summary["total"] = workload_summary.drop(columns=["workload"]).sum(axis=1)
display(workload_summary.sort_values("total", ascending=False).head(20))

fig, ax = plt.subplots(figsize=(10, 4))
plot_data = workload_summary.sort_values("total", ascending=False)
ax.bar(plot_data["workload"], plot_data.get("normal", 0), label="Норма")
ax.bar(plot_data["workload"], plot_data.get("failure", 0), bottom=plot_data.get("normal", 0), label="Сбой")
ax.set_title("Покрытие STRICT-запусков по workload")
ax.set_ylabel("Кол-во запусков")
ax.set_xlabel("Workload")
ax.tick_params(axis="x", rotation=60)
ax.legend()
plt.tight_layout()
plt.show()


**Что показано.** Количество запусков с двумя модальностями по workload.  
**Что наблюдается.** Данные сгруппированы по повторяющимся workload и сценариям отказа.  
**Локальный вывод.** Split должен быть workload/run-aware, иначе модель может запомнить особенности конкретного workload вместо раннего сигнала деградации.


## 3. Ограничения данных

Финальный пайплайн должен явно учитывать четыре ограничения: несинхронность логов и KPI, пропуски модальностей, дисбаланс целевых окон и риск поздней логовой утечки. Поэтому следующий этап строит строгие окна с `gap10`: модель не видит момент события и получает только предшествующий контекст.


In [ ]:
issue_summary = issues.groupby("issue").size().reset_index(name="count") if not issues.empty else pd.DataFrame(columns=["issue", "count"])
display(issue_summary)

eda_payload = {
    "raw_data": str(RAW_DATA.relative_to(PROJECT_ROOT)),
    "strict_runs": int(strict["run_id"].nunique()),
    "workloads": int(strict["workload"].nunique()),
    "normal_runs": int(strict.loc[strict["scenario_type"].eq("normal"), "run_id"].nunique()),
    "failure_runs": int(strict.loc[strict["scenario_type"].eq("failure"), "run_id"].nunique()),
    "non_strict_issues": issue_summary.to_dict("records"),
}
write_json(REPORT_DIR / "01_eda_summary.json", eda_payload)
print("Записано:", (REPORT_DIR / "01_eda_summary.json").relative_to(PROJECT_ROOT))


**Handoff в 02.** Следующий ноутбук строит из `raw_data/` строгий window-contract: временная сетка `10s`, контекст логов `60s`, KPI-агрегаты, `strict_lead_sec`, `warning/risk intensity`, split и baseline.
